# 00 — Comparación Raw vs Procesado por Fuente

Evalúa el impacto del preprocesado sobre las predicciones YOLO, **por separado** para cada pipeline:

| Fuente | Raw | Procesado | Ruido eliminado |
|--------|-----|-----------|----------------|
| **Mic fijo** | `data/raw/*.txt` (inferencia sin limpiar) | `predicciones_clean.csv` | Declip + Wiener + Lowpass 7500 Hz |
| **Móvil** | `predictions_mobile_raw.parquet` (pre-NMS) | `predictions_mobile.parquet` (post-NMS) | NMS IoU 0.7 (Wiener gated ya aplicado en ambos) |

> **Nota**: las dos pipelines eliminan tipos de ruido distintos y no son comparables entre sí directamente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats as sp_stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)

CLASSES = ['Horn','Siren','Pets','Physiological','Speech','Ring Tone','Vibrating','Notifications','Cry']
CLR_RAW  = 'tomato'
CLR_PROC = 'steelblue'

DATA_DIR = Path('../data/processed')
RAW_DIR  = Path('../data/raw')
OUT_DIR  = Path('../outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def add_class_name(df, col='class'):
    df = df.copy()
    df['class_name'] = df[col].apply(
        lambda x: CLASSES[int(x)] if pd.notna(x) and 0 <= int(x) < len(CLASSES) else 'Unknown'
    )
    return df

def class_delta_table(df_a, df_b, label_a='Raw', label_b='Proc'):
    ca = df_a.groupby('class_name').size().rename(label_a)
    cb = df_b.groupby('class_name').size().rename(label_b)
    out = pd.DataFrame({label_a: ca, label_b: cb}).reindex(CLASSES, fill_value=0)
    out['Δ'] = out[label_b] - out[label_a]
    out['Δ %'] = (out['Δ'] / out[label_a].replace(0, np.nan) * 100).round(1)
    return out

def plot_class_delta(df_cls, title, fname, label_a='Raw', label_b='Proc'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x, w = np.arange(len(CLASSES)), 0.38
    axes[0].bar(x - w/2, df_cls[label_a], width=w, color=CLR_RAW,  alpha=0.8, label=label_a)
    axes[0].bar(x + w/2, df_cls[label_b], width=w, color=CLR_PROC, alpha=0.8, label=label_b)
    axes[0].set_xticks(x); axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
    axes[0].set_title(f'{title} — Eventos por Clase'); axes[0].legend()
    colors = ['green' if v >= 0 else 'red' for v in df_cls['Δ %'].fillna(0)]
    axes[1].barh(CLASSES, df_cls['Δ %'].fillna(0), color=colors)
    axes[1].axvline(0, color='black', lw=0.8)
    axes[1].set_title(f'{title} — Δ % por Clase'); axes[1].set_xlabel('Δ %')
    for i, v in enumerate(df_cls['Δ %'].fillna(0)):
        axes[1].text(v + (1 if v >= 0 else -1), i, f'{v:+.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, bbox_inches='tight'); plt.show()

def plot_confidence(df_a, df_b, title, fname, label_a='Raw', label_b='Proc'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_a['confidence'].dropna(), bins=40, alpha=0.6, color=CLR_RAW,  label=f'{label_a} (n={len(df_a):,})', density=True)
    axes[0].hist(df_b['confidence'].dropna(), bins=40, alpha=0.6, color=CLR_PROC, label=f'{label_b} (n={len(df_b):,})', density=True)
    axes[0].axvline(df_a['confidence'].mean(), color='red',  ls='--', lw=1.5, label=f'Media {label_a}={df_a["confidence"].mean():.3f}')
    axes[0].axvline(df_b['confidence'].mean(), color='blue', ls='--', lw=1.5, label=f'Media {label_b}={df_b["confidence"].mean():.3f}')
    axes[0].set_title(f'{title} — Confianza Global'); axes[0].set_xlabel('Confianza'); axes[0].legend(fontsize=9)
    combined = pd.concat([
        df_a.assign(fuente=label_a),
        df_b.assign(fuente=label_b),
    ])
    sns.boxplot(data=combined, x='class_name', y='confidence', hue='fuente',
                palette={label_a: CLR_RAW, label_b: CLR_PROC}, ax=axes[1],
                order=CLASSES, fliersize=2)
    axes[1].set_title(f'{title} — Confianza por Clase'); axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, bbox_inches='tight'); plt.show()

print('Setup OK')

---
## Parte A — Micrófonos Fijos
**Raw**: inferencia directa sobre audio sin procesar (`data/raw/**/predicciones*.txt`)  
**Clean**: inferencia sobre audio procesado con Declip + Wiener + Lowpass 7500 Hz (`predicciones_clean.csv`)

In [ ]:
# --- Raw mic (predicciones*.txt por día) ---
COLS_RAW = ['microfono_id', 't_start', 't_end', 'class', 'confidence']
dfs_raw = []
for f in RAW_DIR.rglob('predicciones*.txt'):
    df = pd.read_csv(f, header=None, names=COLS_RAW, on_bad_lines='skip')
    df['date'] = f.parent.name
    dfs_raw.append(df)

if not dfs_raw:
    print('[WARN] No se encontraron predicciones*.txt en data/raw/ — Parte A omitida')
    mic_raw_ok = False
else:
    mic_raw = pd.concat(dfs_raw, ignore_index=True)
    mic_raw['t_start'] = pd.to_datetime(mic_raw['t_start'], format='mixed', errors='coerce')
    mic_raw = mic_raw.dropna(subset=['t_start'])
    mic_raw['class'] = pd.to_numeric(mic_raw['class'], errors='coerce').astype('Int64')
    mic_raw = add_class_name(mic_raw)
    mic_raw_ok = True

# --- Clean mic (predicciones_clean.csv) ---
clean_csv = DATA_DIR / 'predicciones_clean.csv'
if not clean_csv.exists():
    print(f'[WARN] {clean_csv.name} no encontrado — ejecutar scripts/infer_clean.py')
    mic_clean_ok = False
else:
    mic_clean = pd.read_csv(clean_csv)
    mic_clean = mic_clean.rename(columns={
        'mic_id':'microfono_id','timestamp_onset':'t_start',
        'timestamp_offset':'t_end','class_id':'class'
    })
    mic_clean['t_start'] = pd.to_datetime(mic_clean['t_start'], format='mixed', errors='coerce')
    mic_clean = mic_clean.dropna(subset=['t_start'])
    mic_clean['date'] = mic_clean['t_start'].dt.strftime('%d-%m-%Y')
    mic_clean['class'] = pd.to_numeric(mic_clean['class'], errors='coerce').astype('Int64')
    mic_clean = add_class_name(mic_clean)
    mic_clean_ok = True

if mic_raw_ok and mic_clean_ok:
    shared_mic = sorted(set(mic_raw['date'].unique()) & set(mic_clean['date'].unique()))
    print(f'Mic RAW   : {len(mic_raw):,} eventos en {mic_raw["date"].nunique()} días')
    print(f'Mic CLEAN : {len(mic_clean):,} eventos en {mic_clean["date"].nunique()} días')
    print(f'Días compartidos: {len(shared_mic)} → {shared_mic}')
    print(f'Solo RAW:  {sorted(set(mic_raw["date"].unique()) - set(mic_clean["date"].unique()))}')
    print(f'Solo CLEAN:{sorted(set(mic_clean["date"].unique()) - set(mic_raw["date"].unique()))}')

### A1. Cobertura de Fechas

In [ ]:
if not (mic_raw_ok and mic_clean_ok):
    print('Datos mic no disponibles'); raise SystemExit()

all_dates = sorted(set(mic_raw['date'].unique()) | set(mic_clean['date'].unique()))
df_dates = pd.DataFrame({
    'raw':   mic_raw.groupby('date').size(),
    'clean': mic_clean.groupby('date').size(),
}).reindex(all_dates).fillna(0).astype(int)
df_dates['Δ'] = df_dates['clean'] - df_dates['raw']
df_dates['Δ %'] = (df_dates['Δ'] / df_dates['raw'].replace(0, np.nan) * 100).round(1)
display(df_dates)

fig, ax = plt.subplots(figsize=(13, 3))
x = range(len(all_dates))
ax.bar(x, df_dates['raw'],   label='Raw',   alpha=0.6, color=CLR_RAW)
ax.bar(x, df_dates['clean'], label='Clean', alpha=0.7, color=CLR_PROC, width=0.5)
ax.set_xticks(list(x))
ax.set_xticklabels([d[:-5] for d in all_dates], rotation=45, ha='right')
ax.set_title('Mic — Eventos por Día: Raw vs Clean'); ax.set_ylabel('Nº eventos'); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'mic_coverage.png', bbox_inches='tight'); plt.show()

### A2. Impacto Total (días compartidos)

In [ ]:
mic_r = mic_raw[mic_raw['date'].isin(shared_mic)].copy()
mic_c = mic_clean[mic_clean['date'].isin(shared_mic)].copy()

delta = len(mic_c) - len(mic_r)
delta_pct = delta / len(mic_r) * 100
t_stat, p_val = sp_stats.mannwhitneyu(mic_c['confidence'].dropna(), mic_r['confidence'].dropna(), alternative='two-sided')

print(f'=== MIC — {len(shared_mic)} días compartidos ===')
print(f'Raw   : {len(mic_r):,} eventos | Conf media: {mic_r["confidence"].mean():.4f}')
print(f'Clean : {len(mic_c):,} eventos | Conf media: {mic_c["confidence"].mean():.4f}')
print(f'Δ eventos : {delta:+,} ({delta_pct:+.1f}%)')
print(f'Δ confianza: {mic_c["confidence"].mean() - mic_r["confidence"].mean():+.4f}')
print(f'Mann-Whitney p={p_val:.2e} → {"diferencia significativa" if p_val < 0.05 else "sin diferencia significativa"}')

### A3. Cambio por Clase

In [ ]:
df_cls_mic = class_delta_table(mic_r, mic_c, label_a='Raw', label_b='Clean')
display(df_cls_mic)
plot_class_delta(df_cls_mic, 'Mic', 'mic_class_delta.png', label_a='Raw', label_b='Clean')

### A4. Distribución de Confianza

In [ ]:
plot_confidence(mic_r, mic_c, 'Mic', 'mic_confidence.png', label_a='Raw', label_b='Clean')

### A5. Análisis por Día

In [ ]:
raw_daily   = mic_r.groupby('date')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_raw','count':'n_raw'})
clean_daily = mic_c.groupby('date')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_clean','count':'n_clean'})
daily = pd.concat([raw_daily, clean_daily], axis=1).reindex(sorted(shared_mic))
daily['Δ conf'] = (daily['conf_clean'] - daily['conf_raw']).round(4)
daily['Δ N %']  = ((daily['n_clean'] - daily['n_raw']) / daily['n_raw'] * 100).round(1)
display(daily.rename(columns={'conf_raw':'Conf Raw','conf_clean':'Conf Clean','n_raw':'N Raw','n_clean':'N Clean'}))

dates_short = [d[:-5] for d in sorted(shared_mic)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(dates_short, daily['Δ N %'],
            color=['green' if x >= 0 else 'red' for x in daily['Δ N %']])
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Mic — Δ % Eventos por Día'); axes[0].set_ylabel('Δ %')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(dates_short, daily['Δ conf'],
            color=['green' if x >= 0 else 'red' for x in daily['Δ conf']])
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Mic — Δ Confianza Media por Día'); axes[1].set_ylabel('Δ Confianza')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / 'mic_daily.png', bbox_inches='tight'); plt.show()

# Zoom: día con mayor reducción
worst = daily['Δ N %'].idxmin()
print(f'\nDía con mayor reducción: {worst} ({daily.loc[worst, "Δ N %"]:.1f}%)')
r_d = mic_r[mic_r['date'] == worst]
c_d = mic_c[mic_c['date'] == worst]
df_day = class_delta_table(r_d, c_d, label_a='Raw', label_b='Clean')
display(df_day)
plot_class_delta(df_day, worst, f'mic_zoom_{worst}.png', label_a='Raw', label_b='Clean')

---
## Parte B — Móvil
**Sin Wiener**: inferencia sobre audio crudo convertido a WAV 16kHz, sin filtrado (`predictions_mobile_noWiener.parquet`)  
**Wiener Gated**: inferencia sobre audio filtrado SNR-gate 8dB, hp=100Hz, nr=0.50, ×2 (`predictions_mobile.parquet`)

Ambos post-NMS IoU ≥ 0.7 → comparación equivalente a Parte A (efecto del filtrado de audio).

In [ ]:
no_wiener_path = DATA_DIR / 'predictions_mobile_noWiener.parquet'
proc_mob_path  = DATA_DIR / 'predictions_mobile.parquet'

mob_nw_ok   = no_wiener_path.exists()
mob_proc_ok = proc_mob_path.exists()

if not mob_nw_ok:
    print(f'[WARN] {no_wiener_path.name} no encontrado — ejecutar scripts/prepare_mobile.py')
if not mob_proc_ok:
    print(f'[WARN] {proc_mob_path.name} no encontrado — ejecutar scripts/prepare_mobile.py')

def _load_mobile_pq(path):
    df = pd.read_parquet(path)
    df = df.rename(columns={
        'timestamp_onset':'t_start', 'timestamp_offset':'t_end', 'class_id':'class'
    })
    df['t_start'] = pd.to_datetime(df['t_start'], utc=True)
    df['class']   = pd.to_numeric(df['class'], errors='coerce').astype('Int64')
    df = add_class_name(df)
    return df

if mob_nw_ok and mob_proc_ok:
    mob_nw   = _load_mobile_pq(no_wiener_path)
    mob_proc = _load_mobile_pq(proc_mob_path)
    shared_sessions = sorted(set(mob_nw['session_id'].unique()) & set(mob_proc['session_id'].unique()))
    print(f'Móvil Sin Wiener  : {len(mob_nw):,} eventos en {mob_nw["session_id"].nunique()} sesiones')
    print(f'Móvil Wiener Gated: {len(mob_proc):,} eventos en {mob_proc["session_id"].nunique()} sesiones')
    print(f'Sesiones compartidas: {len(shared_sessions)} → {shared_sessions}')

### B1. Cobertura por Sesión

In [ ]:
if not (mob_nw_ok and mob_proc_ok):
    print('Datos móvil no disponibles'); raise SystemExit()

all_sessions = sorted(set(mob_nw['session_id'].unique()) | set(mob_proc['session_id'].unique()))
df_sess = pd.DataFrame({
    'Sin Wiener':   mob_nw.groupby('session_id').size(),
    'Wiener Gated': mob_proc.groupby('session_id').size(),
}).reindex(all_sessions).fillna(0).astype(int)
df_sess['Δ'] = df_sess['Wiener Gated'] - df_sess['Sin Wiener']
df_sess['Δ %'] = (df_sess['Δ'] / df_sess['Sin Wiener'].replace(0, np.nan) * 100).round(1)
display(df_sess)

fig, ax = plt.subplots(figsize=(13, 3))
x = range(len(all_sessions))
ax.bar(x, df_sess['Sin Wiener'],   label='Sin Wiener',   alpha=0.6, color=CLR_RAW)
ax.bar(x, df_sess['Wiener Gated'], label='Wiener Gated', alpha=0.7, color=CLR_PROC, width=0.5)
ax.set_xticks(list(x)); ax.set_xticklabels(all_sessions, rotation=45, ha='right')
ax.set_title('Móvil — Eventos por Sesión: Sin Wiener vs Wiener Gated'); ax.set_ylabel('Nº eventos'); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'mob_coverage.png', bbox_inches='tight'); plt.show()

### B2. Impacto Total (sesiones compartidas)

In [ ]:
mob_r = mob_nw[mob_nw['session_id'].isin(shared_sessions)].copy()
mob_p = mob_proc[mob_proc['session_id'].isin(shared_sessions)].copy()

delta = len(mob_p) - len(mob_r)
delta_pct = delta / len(mob_r) * 100
t_stat, p_val = sp_stats.mannwhitneyu(mob_p['confidence'].dropna(), mob_r['confidence'].dropna(), alternative='two-sided')

print(f'=== MÓVIL — {len(shared_sessions)} sesiones compartidas ===')
print(f'Sin Wiener   : {len(mob_r):,} eventos | Conf media: {mob_r["confidence"].mean():.4f}')
print(f'Wiener Gated : {len(mob_p):,} eventos | Conf media: {mob_p["confidence"].mean():.4f}')
print(f'Δ eventos    : {delta:+,} ({delta_pct:+.1f}%)')
print(f'Δ confianza  : {mob_p["confidence"].mean() - mob_r["confidence"].mean():+.4f}')
print(f'Mann-Whitney p={p_val:.2e} → {"diferencia significativa" if p_val < 0.05 else "sin diferencia significativa"}')

### B3. Cambio por Clase

In [ ]:
df_cls_mob = class_delta_table(mob_r, mob_p, label_a='Sin Wiener', label_b='Wiener Gated')
display(df_cls_mob)
plot_class_delta(df_cls_mob, 'Móvil', 'mob_class_delta.png', label_a='Sin Wiener', label_b='Wiener Gated')

### B4. Distribución de Confianza

In [ ]:
plot_confidence(mob_r, mob_p, 'Móvil', 'mob_confidence.png', label_a='Sin Wiener', label_b='Wiener Gated')

### B5. Análisis por Sesión

In [ ]:
raw_sess  = mob_r.groupby('session_id')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_sinW','count':'n_sinW'})
proc_sess = mob_p.groupby('session_id')['confidence'].agg(['mean','count']).rename(columns={'mean':'conf_wiener','count':'n_wiener'})
sess = pd.concat([raw_sess, proc_sess], axis=1).reindex(sorted(shared_sessions))
sess['Δ conf'] = (sess['conf_wiener'] - sess['conf_sinW']).round(4)
sess['Δ N %']  = ((sess['n_wiener'] - sess['n_sinW']) / sess['n_sinW'] * 100).round(1)
display(sess.rename(columns={
    'conf_sinW':'Conf Sin Wiener','conf_wiener':'Conf Wiener','n_sinW':'N Sin Wiener','n_wiener':'N Wiener'
}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
snames = sorted(shared_sessions)
axes[0].bar(snames, sess['Δ N %'],
            color=['green' if x >= 0 else 'red' for x in sess['Δ N %']])
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Móvil — Δ % Eventos por Sesión (efecto Wiener Gated)')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(snames, sess['Δ conf'],
            color=['green' if x >= 0 else 'red' for x in sess['Δ conf']])
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Móvil — Δ Confianza por Sesión (efecto Wiener Gated)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(OUT_DIR / 'mob_sessions.png', bbox_inches='tight'); plt.show()

---
## Resumen Comparativo
Tabla ejecutiva para la memoria del TFG. Muestra el efecto de cada etapa de procesado por fuente.

In [ ]:
rows = []

if mic_raw_ok and mic_clean_ok:
    t_stat_mic, p_mic = sp_stats.mannwhitneyu(
        mic_c['confidence'].dropna(), mic_r['confidence'].dropna(), alternative='two-sided'
    )
    rows += [
        ('Mic — días con datos raw',          mic_raw['date'].nunique()),
        ('Mic — días con datos clean',         mic_clean['date'].nunique()),
        ('Mic — días compartidos',             len(shared_mic)),
        ('Mic — eventos sin procesar',         f'{len(mic_r):,}'),
        ('Mic — eventos procesados',           f'{len(mic_c):,}'),
        ('Mic — Δ eventos',                    f'{len(mic_c)-len(mic_r):+,} ({(len(mic_c)-len(mic_r))/len(mic_r)*100:+.1f}%)'),
        ('Mic — confianza sin procesar',       f'{mic_r["confidence"].mean():.4f}'),
        ('Mic — confianza procesado',          f'{mic_c["confidence"].mean():.4f}'),
        ('Mic — Δ confianza',                  f'{mic_c["confidence"].mean()-mic_r["confidence"].mean():+.4f}'),
        ('Mic — Mann-Whitney p',               f'{p_mic:.2e}'),
        ('Mic — procesado',                    'Declip + Wiener + Lowpass 7500 Hz'),
    ]

if mob_nw_ok and mob_proc_ok:
    t_stat_mob, p_mob = sp_stats.mannwhitneyu(
        mob_p['confidence'].dropna(), mob_r['confidence'].dropna(), alternative='two-sided'
    )
    rows += [
        ('Móvil — sesiones procesadas',        mob_proc['session_id'].nunique()),
        ('Móvil — eventos sin Wiener',         f'{len(mob_r):,}'),
        ('Móvil — eventos Wiener Gated',       f'{len(mob_p):,}'),
        ('Móvil — Δ eventos (efecto Wiener)',  f'{len(mob_p)-len(mob_r):+,} ({(len(mob_p)-len(mob_r))/len(mob_r)*100:+.1f}%)'),
        ('Móvil — confianza sin Wiener',       f'{mob_r["confidence"].mean():.4f}'),
        ('Móvil — confianza Wiener Gated',     f'{mob_p["confidence"].mean():.4f}'),
        ('Móvil — Δ confianza',                f'{mob_p["confidence"].mean()-mob_r["confidence"].mean():+.4f}'),
        ('Móvil — Mann-Whitney p',             f'{p_mob:.2e}'),
        ('Móvil — procesado audio',            'Wiener gated (SNR-gate 8dB, hp=100Hz, nr=0.50, ×2)'),
        ('Móvil — dedup detecciones',          'NMS 1D IoU ≥ 0.7 (aplicado en ambas versiones)'),
    ]

if rows:
    display(pd.DataFrame(rows, columns=['Métrica', 'Valor']))
else:
    print('[WARN] Sin datos suficientes para el resumen')